# Lab: Shortest Paths with Bellman-Ford and Floyd-Warshall

**Topics:** Bellman-Ford · Floyd-Warshall
**Time:** about 70–90 minutes  **Language:** Python 3.9+ (no extra libraries needed)

## What you will do

| Task | Topic | What you implement | Points |
|------|-------|--------------------|--------|
| 1 | Bellman-Ford | `bellman_ford`, `build_path`, `bellman_ford_early` | 45 |
| 2 | Floyd-Warshall | `floyd_warshall`, `fw_path`, `all_pairs_bf` (cross-check with Task 1) | 45 |
| – | Reflection | written answers at the end | 10 |
| | | **Total** | **100** |

## How this notebook works

1. Read the instruction text above each code cell.
2. Complete every cell marked `# TODO`. Keep the function names and signatures unchanged.
3. Run the **test cell** under each part. It prints ✅ or ❌ for every check and records your points.
4. At the end, run the **score summary** cell, answer the reflection questions, then save and submit.

> The functions start as empty stubs (`return None`), so **Run All** will not crash. You will simply see ❌ until you fill them in.

**Rules:** write your own code; you may use the lecture slides and the Python documentation. Do not change the test cells.

## 0. Setup (run this cell first)

This cell defines `INF`, the example graphs and the small testing helper `check(...)`. You do not need to edit it.

In [ ]:
import random, time

INF = float("inf")

# ---- tiny testing helper (do not edit) ------------------------------------
RESULTS = {}

def check(task, name, test, pts=1):
    """Run test() and record whether it returned True. Exceptions count as a fail."""
    try:
        ok = bool(test())
        err = ""
    except Exception as e:
        ok, err = False, f"  ({type(e).__name__}: {e})"
    RESULTS.setdefault(task, []).append((name, ok, pts))
    print(("✅" if ok else "❌"), f"[{task}] {name} ({pts} pt){err}")

def summary():
    print(f"{'Task':<22}{'Points':>8}{'Max':>6}")
    total = mx = 0
    for task, rows in RESULTS.items():
        got = sum(p for _, ok, p in rows if ok)
        top = sum(p for _, _, p in rows)
        total += got; mx += top
        print(f"{task:<22}{got:>8}{top:>6}")
    print("-" * 36)
    print(f"{'AUTOMATIC TOTAL':<22}{total:>8}{mx:>6}")

# ---- graphs used in the lab (do not edit) ----------------------------------
V = ["S", "A", "B", "C", "D"]
E = [("S", "A", 6), ("S", "B", 3), ("B", "A", -2),
     ("A", "C", 3), ("B", "D", 7), ("C", "D", -1)]          # has a negative edge, no negative cycle

V_CYCLE = ["S", "A", "B"]
E_CYCLE = [("S", "A", 2), ("A", "B", 2), ("B", "S", -5)]    # cycle S->A->B->S costs -1

def random_graph(n, m, seed=0):
    """Random directed graph with some negative edges but NO negative cycle."""
    rnd = random.Random(seed)
    pot = [rnd.randint(0, 10) for _ in range(n)]
    edges = {}
    while len(edges) < m:
        u, v = rnd.sample(range(n), 2)
        edges[(u, v)] = rnd.randint(1, 9) + pot[u] - pot[v]
    return [(u, v, w) for (u, v), w in edges.items()]

print("Setup done.")

Setup done.


---
# Task 1 · Bellman-Ford (45 points)

## Background

Bellman-Ford finds the cheapest path from one **source** to every vertex, even when some edges have **negative** weights.

It works in rounds. After round $k$, `dist[b]` is the cost of the cheapest path from the source to `b` that uses **at most $k$ edges**:

$$d^{(k)}[b] = \min\Big\{\, d^{(k-1)}[b],\ \min_{(a,b)\in E}\big( d^{(k-1)}[a] + w(a,b) \big) \Big\}$$

* A simple path has at most $n-1$ edges, so $n-1$ rounds are enough when there is no negative cycle.
* If an edge can **still** be improved after $n-1$ rounds, a negative cycle is reachable from the source.
* **Key rule:** in round $k$ read only the *old* values ($k-1$) and write the *new* ones.

### Our graph `V`, `E` (source `S`)

| From | To | Weight |
|------|----|--------|
| S | A | 6 |
| S | B | 3 |
| B | A | −2 |
| A | C | 3 |
| B | D | 7 |
| C | D | −1 |

## Part 1A · Trace by hand (no code)

Before coding, fill in this table on paper or in the cell below (use `inf` for ∞). Each row is one round.

| round | S | A | B | C | D |
|-------|---|---|---|---|---|
| d(0)  | 0 | inf | inf | inf | inf |
| d(1)  | ? | ? | ? | ? | ? |
| d(2)  | ? | ? | ? | ? | ? |
| d(3)  | ? | ? | ? | ? | ? |
| d(4)  | ? | ? | ? | ? | ? |

Write your answers here (a Python list per round is fine):

In [ ]:
my_trace = {
    1: [0, 6, 3, float("inf"), float("inf")],
    2: [0, 1, 3, 9, 10],
    3: [0, 1, 3, 4, 8],
    4: [0, 1, 3, 4, 3],
}

## Part 1B · Implement `bellman_ford`

**Instructions**

1. Create `dist` (all `INF`, source `0`) and `prev` (all `None`).
2. Create `history = [dist.copy()]`; it stores $d^{(0)}, d^{(1)}, \dots$ for printing and testing.
3. Repeat `len(vertices) - 1` rounds. In each round:
   * `new = dist.copy()` (this keeps the "fewer edges" case),
   * for every edge `(a, b, w)`: if `dist[a] + w < new[b]`, set `new[b]` and `prev[b] = a`,
   * after the edge loop set `dist = new` and append `dist.copy()` to `history`.
4. After all rounds, loop over the edges once more. If any edge still improves (`dist[a] + w < dist[b]`), `raise ValueError("negative cycle reachable from source")`.
5. `return dist, prev, history`.

**Hint:** compare against `dist[a]` (old values) but write into `new[b]`. If you write into `dist` directly you have changed the algorithm (it would still often work, but your `history` rows will not match the tests).

In [ ]:
def bellman_ford(vertices, edges, source):
    INF = float("inf")

    dist = {v: INF for v in vertices}
    dist[source] = 0
    prev = {v: None for v in vertices}
    history = [dist.copy()]

    for _ in range(len(vertices) - 1):
        new = dist.copy()
        for a, b, w in edges:
            if dist[a] + w < new[b]:
                new[b] = dist[a] + w
                prev[b] = a
        dist = new
        history.append(dist.copy())

    for a, b, w in edges:
        if dist[a] + w < dist[b]:
            raise ValueError("negative cycle reachable from source")

    return dist, prev, history

## Part 1C · Rebuild a path

`prev[v]` stores the vertex just before `v` on the best path. Starting at the target, follow `prev` back to the source and reverse the list.

Implement `build_path(prev, target)` so that `build_path(prev, "D")` returns `["S", "B", "A", "C", "D"]`.

In [ ]:
def build_path(prev, target):
    path = []
    v = target
    while v is not None:
        path.append(v)
        v = prev[v]
    path.reverse()
    return path

## Part 1D · Early exit (challenge)

If a round changes nothing, later rounds will not change anything either, so we can stop early.

Write `bellman_ford_early(vertices, edges, source)` that behaves like `bellman_ford` but:

* stops **right after the first round in which no distance changed**,
* returns `(dist, prev, rounds)` where `rounds` is the number of rounds that were executed (including the final no-change round),
* still raises `ValueError` for a negative cycle (do the extra check pass if it never stopped early).

*Example:* on a 6-vertex chain `S→A→B` (other vertices unreachable) round 3 changes nothing, so `rounds == 3` even though $n-1 = 5$.

In [ ]:
def bellman_ford_early(vertices, edges, source):
    INF = float("inf")

    dist = {v: INF for v in vertices}
    dist[source] = 0
    prev = {v: None for v in vertices}
    rounds = 0
    stopped_early = False

    for _ in range(len(vertices) - 1):
        new = dist.copy()
        for a, b, w in edges:
            if dist[a] + w < new[b]:
                new[b] = dist[a] + w
                prev[b] = a
        rounds += 1
        changed = new != dist
        dist = new
        if not changed:
            stopped_early = True
            break

    if not stopped_early:
        for a, b, w in edges:
            if dist[a] + w < dist[b]:
                raise ValueError("negative cycle reachable from source")

    return dist, prev, rounds

### Test Task 1 (run this cell)

In [ ]:
T = "Task 1"
expected_hist = [
    [0, INF, INF, INF, INF], [0, 6, 3, INF, INF], [0, 1, 3, 9, 10],
    [0, 1, 3, 4, 8], [0, 1, 3, 4, 3],
]
check(T, "my_trace matches d(1)…d(4)", lambda: [my_trace[k] for k in (1, 2, 3, 4)] == expected_hist[1:], 6)
check(T, "final distances are correct",
      lambda: bellman_ford(V, E, "S")[0] == {"S": 0, "A": 1, "B": 3, "C": 4, "D": 3}, 9)
check(T, "history rows d(0)…d(4) are correct",
      lambda: [[h[v] for v in V] for h in bellman_ford(V, E, "S")[2]] == expected_hist, 7)
check(T, "build_path(prev, 'D') == S,B,A,C,D",
      lambda: build_path(bellman_ford(V, E, "S")[1], "D") == ["S", "B", "A", "C", "D"], 6)
check(T, "build_path(prev, 'S') == ['S']",
      lambda: build_path(bellman_ford(V, E, "S")[1], "S") == ["S"], 2)

def _raises(fn):
    try:
        fn()
    except ValueError:
        return True
    return False

check(T, "negative cycle raises ValueError", lambda: _raises(lambda: bellman_ford(V_CYCLE, E_CYCLE, "S")), 6)

V6 = ["S", "A", "B", "C", "D", "E"]
E6 = [("S", "A", 1), ("A", "B", 1)]
check(T, "early exit: same distances", lambda: bellman_ford_early(V, E, "S")[0] == bellman_ford(V, E, "S")[0], 3)
check(T, "early exit: stops after 3 rounds on the chain graph", lambda: bellman_ford_early(V6, E6, "S")[2] == 3, 4)
check(T, "early exit: still detects negative cycle", lambda: _raises(lambda: bellman_ford_early(V_CYCLE, E_CYCLE, "S")), 2)

✅ [Task 1] my_trace matches d(1)…d(4) (6 pt)
✅ [Task 1] final distances are correct (9 pt)
✅ [Task 1] history rows d(0)…d(4) are correct (7 pt)
✅ [Task 1] build_path(prev, 'D') == S,B,A,C,D (6 pt)
✅ [Task 1] build_path(prev, 'S') == ['S'] (2 pt)
✅ [Task 1] negative cycle raises ValueError (6 pt)
✅ [Task 1] early exit: same distances (3 pt)
✅ [Task 1] early exit: stops after 3 rounds on the chain graph (4 pt)
✅ [Task 1] early exit: still detects negative cycle (2 pt)
